# 01 — Data engine

Stage 1 deliverable: sanity-view the factor-labeled map dataset the β-VAE trains on.
Loads the committed generation config + the generated `data/raw/`, shows factor
distributions, the correlation matrix (the pre-registered gate), a sample grid, and
the determinism note. See `plan.md` Stage 1 and `HYPOTHESES.md`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from fantasy_maps import audit, factors, dataset

a = audit.audit_dataset()
print('records', a.n_records, 'complete', a.complete, 'gate_ok', a.gate_ok)
a.summary()['worst_correlated_pair']

## Factor distributions
Each ground-truth factor should span a real range (the coverage gate).

In [ ]:
F = a.factors; M = a.factor_matrix
fig, axes = plt.subplots(1, len(F), figsize=(4*len(F), 3))
for j, name in enumerate(F):
    axes[j].hist(M[:, j], bins=60, color='#3b6ea5')
    axes[j].set_title(name, fontsize=9); axes[j].set_yticks([])
plt.tight_layout(); plt.show()

## Factor correlation matrix
Pre-registered ceiling |ρ| ≤ 0.8 (`configs/generation.yaml`). Redesigned set — see HYPOTHESES.md amendment 1.

In [ ]:
C = a.corr
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(C, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(F))); ax.set_xticklabels(F, rotation=90, fontsize=8)
ax.set_yticks(range(len(F))); ax.set_yticklabels(F, fontsize=8)
for i in range(len(F)):
    for j in range(len(F)):
        ax.text(j, i, f'{C[i,j]:.2f}', ha='center', va='center', fontsize=8)
fig.colorbar(im); ax.set_title(f'max off-diag |rho| = {a.max_abs_offdiag:.3f}'); plt.show()

## Sample maps
Clean heightmap renders (no labels/political layers), downsampled to the training size.

In [ ]:
sp = dataset.make_splits()
ds = dataset.MapDataset(sp['train'][:16], img_size=128, factor_table=a and dataset.load_factor_table())
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for ax, i in zip(axes.ravel(), range(16)):
    img, fac, seed = ds[i]
    ax.imshow(img.permute(1, 2, 0).numpy()); ax.set_title(f'seed {seed}', fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

## Determinism
Same seed + pinned snapshot → identical factors and near-identical pixels (AA jitter only).
Verified in `tests/test_generation.py::test_determinism_same_seed` (browser gate). Splits are
deterministic and fingerprinted:

In [ ]:
print('split fingerprint:', dataset.split_fingerprint(sp))
print('train/val/eval:', len(sp['train']), len(sp['val']), len(sp['eval']))